Sales Forecasting with PySpark Using Online Retail Data

Forecasting product demand is a crucial step in inventory planning for any e-commerce business.
In this project, I analyze the Online Retail dataset and build a PySpark-based forecasting model that predicts the daily quantity of products sold.

The workflow includes:

Cleaning & transforming transactional retail data

Aggregating sales into daily intervals

Feature engineering (date components + categorical encodings)

Training a Random Forest Regressor

Evaluating accuracy using Mean Absolute Error (MAE)

Forecasting sales volume for Week 39 of 2011

In [ ]:
1. Import Libraries

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, to_date, dayofmonth, month, year,
    weekofyear, dayofweek
)
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
2. Initialize Spark & Load Dataset

spark = SparkSession.builder.appName("OnlineRetailForecast").getOrCreate()

sales_df = spark.read.csv(
    "Online Retail.csv",
    header=True,
    inferSchema=True
)

In [ ]:
3. Preprocess & Extract Date Features

sales_df = sales_df.withColumn(
    "InvoiceDate",
    to_date(to_timestamp(col("InvoiceDate"), "d/M/yyyy H:mm"))
)

sales_df = (
    sales_df
    .withColumn("Year", year("InvoiceDate"))
    .withColumn("Month", month("InvoiceDate"))
    .withColumn("Day", dayofmonth("InvoiceDate"))
    .withColumn("Week", weekofyear("InvoiceDate"))
    .withColumn("DayOfWeek", dayofweek("InvoiceDate"))
)

In [ ]:
4. Aggregate Sales into Daily Level

daily_df = (
    sales_df.groupBy(
        "Country", "StockCode", "InvoiceDate",
        "Year", "Month", "Day", "Week", "DayOfWeek"
    )
    .agg({"Quantity": "sum", "UnitPrice": "avg"})
    .withColumnRenamed("sum(Quantity)", "Quantity")
)

In [ ]:
5. Train/Test Split (before and after 2011-09-25)

split_date = "2011-09-25"

train_df = daily_df.filter(col("InvoiceDate") <= split_date)
test_df  = daily_df.filter(col("InvoiceDate") > split_date)

pd_daily_train_data = train_df.toPandas()   # required output

In [ ]:
6. Feature Engineering + Pipeline Setup

country_idx = StringIndexer(
    inputCol="Country", outputCol="CountryIndex"
).setHandleInvalid("keep")

stock_idx = StringIndexer(
    inputCol="StockCode", outputCol="StockCodeIndex"
).setHandleInvalid("keep")

feature_cols = [
    "CountryIndex", "StockCodeIndex",
    "Month", "Year", "DayOfWeek", "Day", "Week"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

rf_model = RandomForestRegressor(
    featuresCol="features",
    labelCol="Quantity",
    maxBins=4000
)

pipeline = Pipeline(
    stages=[country_idx, stock_idx, assembler, rf_model]
)

In [ ]:
7. Train the Forecasting Model

model = pipeline.fit(train_df)

In [ ]:
8. Predict on Test Set

pred_df = (
    model.transform(test_df)
          .withColumn("prediction", col("prediction").cast("double"))
)

In [ ]:
9. Evaluate Accuracy — Mean Absolute Error (MAE)

evaluator = RegressionEvaluator(
    labelCol="Quantity",
    predictionCol="prediction",
    metricName="mae"
)

mae = evaluator.evaluate(pred_df)
mae

In [ ]:
10. Forecast Week 39 of 2011

weekly_pred = pred_df.groupBy("Year", "Week").agg({"prediction": "sum"})

week_39 = weekly_pred.filter(col("Week") == 39)

quantity_sold_w39 = int(
    week_39.select("sum(prediction)").collect()[0][0]
)

quantity_sold_w39

In [ ]:
11. Stop Spark

spark.stop()